In [6]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.2 MB 12.2 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 12.1 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.2 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 10.2 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.4/36.6 MB 11.2 MB/s eta 0:00:04
   ----- ---------------------------------- 5.0/36.6 MB 12.1 MB/s eta 0:00:03
   ------- -------------------------------- 6.8/36.6 MB 11.0 MB/s eta 0:00:03
   -------- ------------------------------- 7.3/36.6 MB 10.8 MB/s eta 0:00:03
   -------- ------------------------------- 8.1/36.6 MB 8.0 MB/s eta 0:00:04
   ----------- ---------------------------- 10.5/36.6 MB 8.5 MB/s eta 0:00:04
   -------------- ------------------------- 13.1/36.6 MB 9.0 MB/s eta 0:00:03
   --------

In [7]:
"""
D006 - 모두마켓 월간 정제 파이프라인 (VS Code 실행용)

VS Code에서 실행하는 법:
    1) 아래 "# %%" 로 나뉜 블록은 Jupyter 셀처럼 Shift+Enter로 한 블록씩 실행 가능
    2) 또는 터미널에서 그냥:  python d006_my_pipeline.py
"""

# %% 0. 임포트 -----------------------------------------------------------
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import RobustScaler


# %% 0-1. 원본 데이터 준비 (orders) --------------------------------------
# 실제로는 회사에서 CSV를 받아오지만, 여기서는 강의와 동일한 성격의
# "더러운" 데이터를 직접 만들어 orders 로 둡니다.
def make_orders(n: int = 1500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    df = pd.DataFrame({
        "order_id": np.arange(1, n + 1),
        "amount": rng.choice(
            [9_900, 19_900, 29_900, 49_900, 249_900], size=n
        ).astype(float),
        "customer_age": rng.integers(15, 70, size=n).astype(float),
        "quantity": rng.integers(1, 4, size=n),
        "region": rng.choice(["서울", "Seoul", "경기", "부산", "대구"], size=n),
        "membership": rng.choice(["basic", "silver", "gold", "vip", "VIP"], size=n),
        "channel": rng.choice(["web", "app", "APP", "WEB"], size=n),
        "category": rng.choice(["food", "beauty", "electronics", "fashion", "living"], size=n),
    })

    # 결측 주입
    na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
    df.loc[na_idx, "amount"] = np.nan
    na_idx2 = rng.choice(n, size=int(n * 0.02), replace=False)
    df.loc[na_idx2, "customer_age"] = np.nan

    # 이상치 주입
    out_idx = rng.choice(n, size=int(n * 0.01), replace=False)
    df.loc[out_idx, "customer_age"] = 999
    out_idx2 = rng.choice(n, size=int(n * 0.01), replace=False)
    df.loc[out_idx2, "quantity"] = 80

    return df


orders = make_orders()
print("orders 준비 완료:", orders.shape)


# %% 1. 단계 함수: 정제 -----------------------------------------------
def clean_strings(df):
    # 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )


def drop_invalid(df, age_min=0, age_max=120, qty_max=20):
    # 불가능한 값(나이 범위·과대 수량)과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
    )


def add_features(df):
    # 분석에 쓸 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
    )


print("세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.")


# %% 2. 한 줄로 조립 (미니 버전) ----------------------------------------
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)
print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
print(cleaned.head(3))


# %% 3. 인코딩 / 스케일링 함수 추가 --------------------------------------
def encode_categories(df):
    # membership은 Ordinal, region·channel·category는 One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    out = pd.concat(
        [out,
         pd.get_dummies(out["region"], prefix="region", dtype=int),
         pd.get_dummies(out["channel"], prefix="ch", dtype=int),
         pd.get_dummies(out["category"], prefix="cat", dtype=int)],
        axis=1
    )
    return out


def scale_numeric(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 RobustScaler로 스케일링.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in cols], index=df.index)
    return pd.concat([df, scaled_df], axis=1)


# 전체 파이프라인 한 줄 (미니 버전 확장)
pipeline_result = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
    .pipe(encode_categories)
    .pipe(scale_numeric)
)
print("최종 shape:", pipeline_result.shape)
print("새로 생긴 컬럼 일부:",
      [c for c in pipeline_result.columns if "scaled" in c or c.startswith("region_")][:8])


# %% 4. amount_class 추가 예시 -------------------------------------------
def add_amount_class(df):
    return df.assign(
        amount_class=np.where(df["amount"] >= 100_000, "high", "low")
    )


orders_v2 = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid)
    .pipe(add_features)
    .pipe(add_amount_class)
)
print(orders_v2[["amount", "amount_class"]].head())
print(orders_v2["amount_class"].value_counts())


# %% 시나리오 0 — 원본 CSV 파일 준비 -------------------------------------
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"
orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")


# %% 시나리오 1 — 전체용 단계 함수들 (파일 입출력 포함) --------------------
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)


def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )


def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )


def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(
            d["amount"] >= 100_000, "high",
            np.where(d["amount"] >= 30_000, "mid", "low")
        )
    )


def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"], prefix="region", dtype=int),
        pd.get_dummies(out["channel"], prefix="ch", dtype=int),
        pd.get_dummies(out["category"], prefix="cat", dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)


def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(
        scaled, columns=[f"{c}_scaled" for c in cols], index=df.index
    )
    return pd.concat([df, scaled_df], axis=1)


print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")


# %% 시나리오 2 — end-to-end 파이프라인 ----------------------------------
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )


# 진짜 한 줄 호출
clean_df = preprocess(input_csv)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
print(clean_df.head(3))


# %% 시나리오 3 — 저장 + 재로드 검증 -------------------------------------
clean_df.to_csv(output_csv, index=False)
reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)


# %% 품질 리포트 함수 -----------------------------------------------------
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    report = {
        "rows_before": before.shape[0],
        "rows_after": after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after": after.shape[1],
        "new_cols": after.shape[1] - before.shape[1],
        "missing_top5_before": (
            before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report


# 사용
rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

orders 준비 완료: (1500, 8)
세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.
원본: (1500, 8) → 정제 후: (1396, 10)
   order_id   amount  customer_age  quantity region membership channel  \
0         1   9900.0          44.0         3     부산        vip     web   
1         2  49900.0          29.0         3     경기        vip     web   
2         3  49900.0          27.0         3     대구        vip     app   

      category  amount_log  is_premium  
0  electronics    9.200391           1  
1         food   10.817796           1  
2      fashion   10.817796           1  
최종 shape: (1396, 25)
새로 생긴 컬럼 일부: ['region_경기', 'region_대구', 'region_부산', 'region_서울', 'customer_age_scaled', 'amount_scaled', 'quantity_scaled']
    amount amount_class
0   9900.0          low
1  49900.0          low
2  49900.0          low
3  29900.0          low
4  29900.0          low
amount_class
low     1115
high     281
Name: count, dtype: int64
원본 CSV 저장: d006_work\orders_raw.csv — 65985 bytes
단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.
입력 행 수 → 